# 02 · Stage 1 · VLM 微调（图→Markdown 表格）

**模型**：`mlx-community/Qwen2-VL-2B-Instruct-4bit`（4-bit 量化，~3GB）

**框架**：MLX-VLM（Apple Silicon 原生，8GB 也能跑）

**备选**：Colab 用 Unsloth（见文末）


In [1]:
# ⚠️ 前置检查：train.jsonl 不存在就自动帮你跑 prepare_stage1.py
import subprocess, sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from src import config as C

s1 = C.DATA_DIR / 'stage1_train' / 'train.jsonl'
raw = C.RAW_DIR / 'pubtabnet_sample'
if not s1.exists():
    if not raw.exists():
        raise RuntimeError(
            '先跑 `uv run python scripts/download_data.py` 下载数据集'
        )
    print('⏳ stage1_train/train.jsonl 不存在，自动跑 prepare_stage1.py ...')
    subprocess.check_call([sys.executable, str(C.ROOT / 'scripts' / 'prepare_stage1.py')])
print('✅ 就绪:', s1)

✅ 就绪: /Users/pc-rn/ws/ai-lab/ocr-fine-app/data/stage1_train/train.jsonl


## 0. 检查环境

In [2]:
import platform, torch
print('Python:', platform.python_version())
try:
    import mlx.core as mx
    print('MLX:', mx.__version__, '| device:', mx.default_device())
except Exception as e:
    print('MLX 未安装:', e)


Python: 3.11.11
MLX: 0.31.1 | device: Device(gpu, 0)


## 1. 转换数据格式为 MLX-VLM 期望格式

MLX-VLM 的 `lora.py` 期望 jsonl，每行 `{"messages": [...], "images": [path]}`。
我们已经在 `prepare_stage1.py` 里产出 sharegpt 格式，稍作转换。

In [3]:
import json, sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from src import config as C
from src.data import load_jsonl

def to_mlx_vlm(rows):
    out = []
    for r in rows:
        msgs = r['messages']
        img = next(c['image'] for c in msgs[0]['content'] if c['type']=='image')
        text = next(c['text'] for c in msgs[0]['content'] if c['type']=='text')
        out.append({
            'images': [str(C.ROOT / img)],
            'messages': [
                {'role': 'user', 'content': text},
                {'role': 'assistant', 'content': msgs[1]['content']},
            ]
        })
    return out

train = to_mlx_vlm(load_jsonl(C.DATA_DIR/'stage1_train'/'train.jsonl'))
val   = to_mlx_vlm(load_jsonl(C.DATA_DIR/'stage1_train'/'val.jsonl'))
out_dir = C.DATA_DIR / 'stage1_mlx'
out_dir.mkdir(parents=True, exist_ok=True)
for name, rows in [('train', train), ('valid', val)]:
    with (out_dir / f'{name}.jsonl').open('w', encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False)+'\n')
print('done:', out_dir)

done: /Users/pc-rn/ws/ai-lab/ocr-fine-app/data/stage1_mlx


## 2. 启动 LoRA 训练（MLX-VLM CLI）

新版 `mlx_vlm >= 0.3` CLI 改了：用 `--model-path` / `--dataset` / `--lora-rank` / `--output-path`。
在终端执行（或用下方 `!` 运行）：

```bash
uv run python -m mlx_vlm.lora \
    --model-path mlx-community/Qwen2-VL-2B-Instruct-4bit \
    --dataset data/stage1_mlx \
    --iters 300 \
    --batch-size 1 \
    --lora-rank 8 \
    --lora-alpha 16 \
    --learning-rate 1e-4 \
    --output-path models/stage1_adapter
```

⚠️ `--output-path` 是**保存**路径（首次训练用这个）；`--adapter-path` 是**恢复**路径（用来接着已训的 LoRA 继续训）。首次就填 `--adapter-path` 会报 `FileNotFoundError`。

可选：`--train-vision`（连 vision encoder 一起训，更慢但更准）、`--train-on-completions`（只在 assistant 段算 loss）。

**预期显存**：~6–8GB；300 iter 在 M2 Pro 约 20–30 分钟。

In [ ]:
# 可选：在 notebook 内直接跑（会阻塞 kernel）
# !uv run python -m mlx_vlm.lora \
#     --model-path mlx-community/Qwen2-VL-2B-Instruct-4bit \
#     --dataset ../data/stage1_mlx \
#     --iters 100 --batch-size 1 --lora-rank 8 --lora-alpha 16 \
#     --learning-rate 1e-4 --output-path ../models/stage1_adapter

## 3. 推理对比（微调前 vs 微调后）

In [6]:
from src.infer import extract_table_from_image
import os

img = sorted((C.DATA_DIR/'stage1_images').glob('*.png'))[0]
print('=== 微调前 ===')
print(extract_table_from_image(img, adapter=None, max_tokens=256))
print('\n=== 微调后 ===')
print(extract_table_from_image(img, adapter=str(C.STAGE1_ADAPTER), max_tokens=256))

=== 微调前 ===


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

ImportError: 
Qwen2VLVideoProcessor requires the Torchvision library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


## 4. Colab 备选方案

若本地太慢，打开 `notebooks/02b_stage1_colab.ipynb`（待补，用 Unsloth + bitsandbytes 4bit）。